In [ ]:
!pip install langchain langgraph langchain_openai langchain_groq --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.7/126.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 4.6 MB/s eta 0:00:00


In [ ]:
pip install streamlit --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPEN_API_KEY')
os.environ['NGROK_AUTH_TOKEN'] = userdata.get('NGROK_AUTH_TOKEN')

In [ ]:
pip install pyngrok

In [ ]:
%%writefile app.py
import streamlit as st
import os
import time
from typing_extensions import TypedDict, Optional
import base64
from io import BytesIO

# Import ChatModels
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Import Langgraph
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode

class State(TypedDict):
    topic: str
    code: Optional[str]
    feedback: Optional[str]
    iterations: int
    specialization: Optional[str]
    history: Optional[str]
    rating: Optional[str]
    code_compare: Optional[str]
    original_code: Optional[str]

# Create the LangGraph workflow definition
def create_workflow_graph():
    # Workflow
    workflow = StateGraph(State)

    # Add Nodes
    workflow.add_node("Coder", lambda state: {})  # Placeholder functions for visualization
    workflow.add_node("Reviewer", lambda state: {})
    workflow.add_node("Code Update", lambda state: {})
    workflow.add_node("Approve Results", lambda state: {})

    # Add Edges
    workflow.add_edge(START, "Coder")
    workflow.add_edge("Coder", "Reviewer")
    workflow.add_edge("Code Update", "Reviewer")
    workflow.add_conditional_edges(
        "Reviewer",
        lambda state: "Code Update" if state.get('iterations', 0) < 3 else "Approve Results",
        {
            "Code Update": "Code Update",
            "Approve Results": "Approve Results",
        }
    )
    workflow.add_edge("Approve Results", END)

    return workflow.compile()

# Set up the LLM models
def initialize_llms():
    # Check if we're in a streamlit environment with secrets
    os.environ["OPENAI_API_KEY"] = os.environ['OPENAI_API_KEY']

    return ChatOpenAI(model="gpt-4o")

# Define the agent functions - same as your original code
def coder_agent(state: State, llm_code):
    """
    LLM Agent to code the given function in the specific language
    """
    coder_prompt = "You are a Coder specialized in {}.\
                Generate the code for the following query: {} \
                Output just the code and nothing else."

    topic = state['topic']
    specialization = state['specialization']
    response = llm_code.invoke(coder_prompt.format(specialization, topic))

    return {
        'history': "----Coder--- \n" + response.content,
        'original_code': response.content,
        'code': response.content,
        'iterations': state['iterations']
    }

def reviewer_agent(state: State, llm_code):
    """
    LLM Agent to review the code
    """
    reviewer_prompt = "You are Code reviewer specialized in {}.\
    You need to review the given code following PEP8 guidelines and potential bugs\
    Also check if docstring is added or not, if not make docstring mandatory\
    and point out issues as bullet list.\
    Also you will not make any changes to the code. You will only provide review on the given code.\
    Code:\n {}"

    code = state['code']
    specialization = state['specialization']
    feedback = llm_code.invoke(reviewer_prompt.format(specialization, code))

    return {
        'history': state.get('history', '') + "\n\n----Reviewer--- \n" + feedback.content,
        "feedback": feedback.content,
        "iterations": state["iterations"] + 1
    }

def handle_coder(state: State, llm_code):
    """
    LLM Agent to improve the code based on feedback
    """
    code_improver = "You are a Coder specialized in {}.\
    Improve the given code given the following guidelines. Guideline:\n {} \n \
    Code:\n {} \n \
    Output just the improved code and nothing else."

    history = state.get('history', '')
    feedback = state.get('feedback', '')
    code = state.get('code', '')
    specialization = state.get('specialization', '')

    code_response = llm_code.invoke(code_improver.format(specialization, feedback, code))

    return {
        'history': history + "\n\n----Coder Improvement--- \n" + code_response.content,
        'code': code_response.content
    }

def check_feedback_agent(state: State, llm_code):
    """
    LLM Agent to check if the feedback has been addressed.
    """
    review = llm_code.invoke(f"""
    Given the following python code: \n\n {state['code']}\n\n
    And the feedback: \n\n {state["feedback"]}
    \n\n Does the code address all the feedback? Answer 'Yes' or 'No'
    """)
    evaluation = "Yes" in review.content

    return {
        "evaluation": evaluation,
        "history": state.get('history', '') + "\n\n----Feedback Check--- \n" + review.content
    }

def handle_result(state: State, llm_code):
    """
    LLM Agent to compare original and final code
    """
    code_comparison = "Compare the two code snippets and rate on a scale of 10 to both. Don't output the codes.\nRevised Code: \n {} \n Original Code: \n {}"

    code1 = state.get('code', '')
    code2 = state.get('original_code', '')

    comparison = llm_code.invoke(code_comparison.format(code1, code2))

    return {
        'history': state.get('history', '') + "\n\n----Final Comparison--- \n" + comparison.content,
        'code_compare': comparison.content
    }

def should_continue(state: State, feedback_check):
    """Determine if the workflow should continue or end"""
    if state['iterations'] >= 2:  # Max iterations
        return "Approve Results"
    if feedback_check["evaluation"]:
        return "Approve Results"
    return "Code Update"

# Function to display the current active step in the workflow
def highlight_active_step(step_name):
    steps = ["Coder", "Reviewer", "Code Update", "Approve Results"]
    html = "<div style='margin-top: 20px;'>"

    for step in steps:
        if step == step_name:
            html += f"<div style='display:inline-block; padding:10px; margin:5px; background-color:#4CAF50; color:white; border-radius:5px;'>{step}</div>"
        else:
            html += f"<div style='display:inline-block; padding:10px; margin:5px; background-color:#f0f0f0; color:#666; border-radius:5px;'>{step}</div>"

    html += "</div>"
    return html

# Streamlit app
def main():
    st.set_page_config(page_title="Code Agent Workflow", layout="wide")

    st.title("Peer Code Review Agentic AI")

    # Sidebar for inputs and workflow diagram
    with st.sidebar:
        st.header("Configuration")
        topic = st.text_area("What would you like the code to do?",
                           "Create a function to find prime numbers up to n using the Sieve of Eratosthenes algorithm.")
        specialization = st.selectbox(
            "Choose coding specialization:",
            ["Python", "JavaScript", "Java", "C++", "Go", "Rust"]
        )

        # Display the LangGraph workflow visualization
        st.subheader("Workflow Diagram")

        # Create and show the graph
        graph = create_workflow_graph()

        # Generate and display the mermaid diagram
        try:
            # Get the mermaid PNG from LangGraph
            png_data = graph.get_graph().draw_mermaid_png()

            # Display the PNG image
            st.image(png_data, caption="LangGraph Workflow", use_container_width=True)
        except Exception as e:
            st.error(f"Could not display graph: {e}")
            # Fallback to text representation
            st.code(graph.get_graph().get_mermaid(), language="mermaid")

        run_button = st.button("Run Workflow")

    # Main area for output in linear sequence
    progress_container = st.container()
    active_step_container = st.container()
    process_container = st.container()
    final_code_container = st.container()
    comparison_container = st.container()
    history_container = st.expander("Full Conversation History", expanded=False)

    # Run the workflow when button is clicked
    if run_button:
        # Initialize progress bar
        progress_bar = progress_container.progress(0)
        status_text = progress_container.empty()
        active_step = active_step_container.empty()

        # Initialize the state
        state = {
            "topic": topic,
            "specialization": specialization,
            "iterations": 0,
            "code": None,
            "feedback": None,
            "history": "",
            "original_code": None,
            "code_compare": None
        }

        # Initialize LLM
        llm_code = initialize_llms()

        # Process container to show each step
        process_section = process_container.container()
        process_section.subheader("Workflow Progress")

        # Step 1: Initial Coding
        status_text.text("Step 1: Generating initial code...")
        active_step.markdown(highlight_active_step("Coder"), unsafe_allow_html=True)
        coder_result = coder_agent(state, llm_code)
        state.update(coder_result)

        process_section.markdown("### Initial Code")
        process_section.code(state["code"], language=specialization.lower())
        progress_bar.progress(20)
        time.sleep(0.5)

        # Loop until done
        while True:
            # Step 2: Review
            status_text.text(f"Step {state['iterations'] + 1}: Reviewing code...")
            active_step.markdown(highlight_active_step("Reviewer"), unsafe_allow_html=True)
            reviewer_result = reviewer_agent(state, llm_code)
            state.update(reviewer_result)

            process_section.markdown(f"### Review - Iteration {state['iterations']}")
            process_section.markdown(state["feedback"])
            progress_bar.progress(40 + 10 * state["iterations"])
            time.sleep(0.5)

            # Step 3: Check Feedback
            status_text.text(f"Step {state['iterations'] + 1}: Checking if feedback is addressed...")
            feedback_check = check_feedback_agent(state, llm_code)
            state.update(feedback_check)
            progress_bar.progress(50 + 10 * state["iterations"])
            time.sleep(0.5)

            # Decide whether to continue
            next_step = should_continue(state, feedback_check)

            if next_step == "Approve Results":
                # Final step
                status_text.text("Finalizing and evaluating code...")
                active_step.markdown(highlight_active_step("Approve Results"), unsafe_allow_html=True)
                result = handle_result(state, llm_code)
                state.update(result)
                progress_bar.progress(100)
                break
            else:
                # Step 4: Update Code
                status_text.text(f"Step {state['iterations'] + 1}: Improving code...")
                active_step.markdown(highlight_active_step("Code Update"), unsafe_allow_html=True)
                coder_update = handle_coder(state, llm_code)
                state.update(coder_update)

                process_section.markdown(f"### Improved Code - Iteration {state['iterations']}")
                process_section.code(state["code"], language=specialization.lower())
                progress_bar.progress(60 + 10 * state["iterations"])
                time.sleep(0.5)

        # Display final results
        status_text.text("✅ Workflow complete!")

        # Final code in a separate section with prominence
        final_code_container.markdown("## 🎯 Final Code")
        final_code_container.code(state["code"], language=specialization.lower())

        # Comparison in its own section
        comparison_container.markdown("## 📊 Code Comparison Analysis")
        comparison_container.markdown(state["code_compare"])

        # Full history in the expander
        history_container.markdown("## Complete Process History")
        history_container.markdown(state["history"])

# Run the Streamlit app
if __name__ == "__main__":
    main()

Overwriting app.py


In [ ]:
!streamlit run app.py --server.port=8990 &>./logs.txt &

In [ ]:
from pyngrok import ngrok
import yaml
import os
# Terminate open tunnels if exist
ngrok.kill()

# Setting the authtoken
# Get your authtoken from `ngrok_credentials.yml` file
# with open('ngrok_credentials.yml', 'r') as file:
#     NGROK_AUTH_TOKEN = yaml.safe_load(file)
ngrok.set_auth_token(os.environ['NGROK_AUTH_TOKEN'] )

# Open an HTTPs tunnel on port XXXX which you get from your `logs.txt` file
ngrok_tunnel = ngrok.connect(8990)
print("Streamlit App:", ngrok_tunnel.public_url)

Streamlit App: https://a799-34-85-213-140.ngrok-free.app
